# Instrument Transposition Prototype

This notebook contains:
- Minimal instrument profiles (range/tessitura/polyphony)
- Two methods: **simple pitch correction (baseline)** vs **constraint-aware correction (advanced)**
- Soft-constraint reporting (tessitura) as metrics
- An evaluation harness to compare methods across instrument categories


In [ ]:
# MIDI pitch numbers (C4 = 60)
INSTRUMENT_PROFILES = {
    # Woodwind
    "FLUTE": {
        "name": "Flute",
        "category": "woodwind",
        "min_pitch": 60,
        "max_pitch": 96,
        "preferred_low": 67,
        "preferred_high": 88,
        "max_polyphony": 1
    },
    "CLARINET": {
        "name": "Clarinet",
        "category": "woodwind",
        "min_pitch": 50,
        "max_pitch": 91,
        "preferred_low": 60,
        "preferred_high": 84,
        "max_polyphony": 1
    },

    # Vocal (example: Tenor)
    "TENOR_VOICE": {
        "name": "Tenor Voice",
        "category": "vocal",
        # Rough singing range; adjust as needed for your project
        "min_pitch": 48,   # C3
        "max_pitch": 69,   # A4
        "preferred_low": 52,  # E3
        "preferred_high": 64, # E4
        "max_polyphony": 1
    },

    # Brasswind (example: Trumpet)
    "TRUMPET": {
        "name": "Trumpet",
        "category": "brasswind",
        # Rough playable range; adjust as needed
        "min_pitch": 55,   # G3
        "max_pitch": 82,   # A#5
        "preferred_low": 60,  # C4
        "preferred_high": 76, # E5
        "max_polyphony": 1
    },

    # Strings (example: Violin)
    "VIOLIN": {
        "name": "Violin",
        "category": "strings",
        "min_pitch": 55,    # G3
        "max_pitch": 103,   # G7
        "preferred_low": 62,  # D4
        "preferred_high": 96, # C7
        "max_polyphony": 1
    },

    # Percussion (pitched) (example: Marimba)
    "MARIMBA": {
        "name": "Marimba",
        "category": "percussion",
        # Many marimbas are 4.3–5 octaves; this is a reasonable placeholder
        "min_pitch": 45,    # A2
        "max_pitch": 93,    # A6
        "preferred_low": 52,  # E3
        "preferred_high": 84, # C6
        "max_polyphony": 4   # mallet instruments can sound multiple notes
    },
}

# Convenience handles
FLUTE = INSTRUMENT_PROFILES["FLUTE"]
CLARINET = INSTRUMENT_PROFILES["CLARINET"]
TENOR_VOICE = INSTRUMENT_PROFILES["TENOR_VOICE"]
TRUMPET = INSTRUMENT_PROFILES["TRUMPET"]
VIOLIN = INSTRUMENT_PROFILES["VIOLIN"]
MARIMBA = INSTRUMENT_PROFILES["MARIMBA"]

## Core functions
Hard constraints: pitch range + polyphony. Soft metric: tessitura violations.


In [ ]:
# Constraint functions
# Each note is represented as (pitch, start_time, duration)

from typing import List, Tuple, Dict, Any, Optional
Note = Tuple[int, float, float]

def out_of_range(notes: List[Note], profile: Dict[str, Any]) -> bool:
    return any(
        (pitch < profile["min_pitch"]) or (pitch > profile["max_pitch"])
        for pitch, _, _ in notes
    )

def max_simultaneous_notes(notes: List[Note]) -> int:
    events = []
    for _, start, dur in notes:
        events.append((start, +1))
        events.append((start + dur, -1))
    events.sort()
    current = 0
    max_poly = 0
    for _, delta in events:
        current += delta
        max_poly = max(max_poly, current)
    return max_poly

def tessitura_violations(notes: List[Note], profile: Dict[str, Any]) -> int:
    return sum(
        (pitch < profile["preferred_low"]) or (pitch > profile["preferred_high"])
        for pitch, _, _ in notes
    )

def shift_notes(notes: List[Note], semitones: int) -> List[Note]:
    return [(pitch + semitones, start, dur) for pitch, start, dur in notes]

## Correction logic
We compare a simple baseline to a constraint-aware method that selects among feasible octave shifts by minimizing tessitura violations.


In [ ]:
# Simple vs advanced (constraint aware) pitch correction

def simple_pitch_correction(notes: List[Note], profile: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simple baseline:
    - Only tries to make notes fit the absolute range via octave shifts.
    - Ignores polyphony and tessitura comfort.
    """
    if not out_of_range(notes, profile):
        return {
            "status": "ACCEPTED",
            "reason": "Baseline: already within range",
            "notes": notes,
            "shift": 0,
            "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
        }

    # Try octave shifts -3..+3; pick the first range-feasible one (baseline behavior)
    for octave in range(-3, 4):
        shifted = shift_notes(notes, octave * 12)
        if not out_of_range(shifted, profile):
            return {
                "status": "CORRECTED",
                "reason": f"Baseline: first-fit octave shift {octave*12} semitones to satisfy range",
                "notes": shifted,
                "shift": octave * 12,
                "soft": {"tessitura_violations": tessitura_violations(shifted, profile)}
            }

    return {
        "status": "REJECTED",
        "reason": "Baseline: no octave shift can satisfy range",
        "notes": None,
        "shift": None,
        "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
    }


def find_best_feasible_transposition(notes: List[Note], profile: Dict[str, Any]) -> Tuple[Optional[List[Note]], Optional[int], Optional[int]]:
    """
    Advanced correction:
    - Enumerate all octave shifts that satisfy hard constraints: range + polyphony.
    - Choose the candidate that minimizes tessitura violations (soft metric),
      tie break by smallest absolute shift.
    Returns: (best_notes, best_shift, best_tessitura_violations)
    """
    candidates = []
    for octave in range(-3, 4):
        shift = octave * 12
        shifted = shift_notes(notes, shift)
        if out_of_range(shifted, profile):
            continue
        if max_simultaneous_notes(shifted) > profile["max_polyphony"]:
            continue
        tv = tessitura_violations(shifted, profile)
        candidates.append((tv, abs(shift), shift, shifted))

    if not candidates:
        return None, None, None

    candidates.sort(key=lambda x: (x[0], x[1]))  # min tessitura, then min |shift|
    tv, _, shift, best = candidates[0]
    return best, shift, tv


def constraint_aware_correction(notes: List[Note], profile: Dict[str, Any]) -> Dict[str, Any]:
    """
    Constraint aware method:
    - Hard constraint 1: polyphony (reject if violated, since we don't rewrite texture yet)
    - Hard constraint 2: range (correctable via octave shifts)
    - Soft signal: tessitura violations (reported, and used for selecting best shift)
    """
    poly = max_simultaneous_notes(notes)
    if poly > profile["max_polyphony"]:
        return {
            "status": "REJECTED",
            "reason": f"Hard constraint: polyphony {poly} exceeds limit {profile['max_polyphony']}",
            "notes": None,
            "shift": None,
            "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
        }

    # If already within range, accept (but still report tessitura)
    if not out_of_range(notes, profile):
        tv = tessitura_violations(notes, profile)
        reason = "Hard constraints satisfied"
        if tv > 0:
            reason += f"; soft warning: {tv} tessitura violations"
        return {
            "status": "ACCEPTED",
            "reason": reason,
            "notes": notes,
            "shift": 0,
            "soft": {"tessitura_violations": tv}
        }

    # Range violated: try best feasible transposition (range+polyphony feasible, minimize tessitura)
    best, shift, tv = find_best_feasible_transposition(notes, profile)
    if best is None:
        return {
            "status": "REJECTED",
            "reason": "Hard constraint: no feasible octave shift satisfies range (and polyphony)",
            "notes": None,
            "shift": None,
            "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
        }

    reason = f"Corrected: chose octave shift {shift} semitones (min tessitura violations)"
    if tv and tv > 0:
        reason += f"; soft warning: {tv} tessitura violations"
    return {
        "status": "CORRECTED",
        "reason": reason,
        "notes": best,
        "shift": shift,
        "soft": {"tessitura_violations": tv}
    }

## Quick test cases


In [ ]:
# Flute tests
melody_out_of_range = [(48, 0.0, 0.5), (50, 0.5, 0.5), (52, 1.0, 0.5)]
chord = [(72, 0.0, 1.0), (76, 0.0, 1.0), (79, 0.0, 1.0)]
valid_melody = [(72, 0.0, 0.5), (74, 0.5, 0.5), (76, 1.0, 0.5)]
boundary_melody = [(FLUTE["min_pitch"], 0.0, 0.5), (FLUTE["max_pitch"], 0.5, 0.5)]
unfixable_melody = [(40, 0.0, 0.5), (100, 0.5, 0.5)]

# Clarinet tests
clarinet_valid = [(62, 0.0, 0.5), (65, 0.5, 0.5), (69, 1.0, 0.5)]
clarinet_out_of_range = [(45, 0.0, 0.5), (47, 0.5, 0.5), (49, 1.0, 0.5)]

# Cross-category sanity: Marimba chord should be acceptable (polyphonic)
marimba_chord = [(60, 0.0, 1.0), (64, 0.0, 1.0), (67, 0.0, 1.0)]

print("Constraint aware (Flute chord):", constraint_aware_correction(chord, FLUTE))
print("Baseline (Flute chord):", simple_pitch_correction(chord, FLUTE))
print("Constraint aware (Marimba chord):", constraint_aware_correction(marimba_chord, MARIMBA))

## Evaluation harness
Runs the same clips across instruments and methods and returns a results table.


In [ ]:
import pandas as pd

METHODS = {
    "simple_pitch": simple_pitch_correction,
    "constraint_aware": constraint_aware_correction,
}

EVAL_CLIPS = {
    "flute_out_of_range": melody_out_of_range,
    "flute_chord": chord,
    "flute_valid": valid_melody,
    "flute_boundary": boundary_melody,
    "flute_unfixable": unfixable_melody,
    "clarinet_valid": clarinet_valid,
    "clarinet_out_of_range": clarinet_out_of_range,
    "marimba_chord": marimba_chord,
}

EVAL_INSTRUMENTS = {
    "FLUTE": FLUTE,
    "CLARINET": CLARINET,
    "TENOR_VOICE": TENOR_VOICE,
    "TRUMPET": TRUMPET,
    "VIOLIN": VIOLIN,
    "MARIMBA": MARIMBA,
}

def run_harness(clips, instruments, methods) -> pd.DataFrame:
    rows = []
    for clip_name, notes in clips.items():
        for inst_key, profile in instruments.items():
            for method_name, fn in methods.items():
                res = fn(notes, profile)
                playable = (res["notes"] is not None)
                rows.append({
                    "clip": clip_name,
                    "instrument": inst_key,
                    "category": profile.get("category", ""),
                    "method": method_name,
                    "status": res.get("status"),
                    "playable": playable,
                    "shift": res.get("shift"),
                    "tessitura_violations": (res.get("soft", {}) or {}).get("tessitura_violations"),
                    "reason": res.get("reason"),
                })
    return pd.DataFrame(rows)

df_results = run_harness(EVAL_CLIPS, EVAL_INSTRUMENTS, METHODS)

df_results.sort_values(["clip", "instrument", "method"]).head(30)

## Metrics used in the evaluation harness

For each *(clip × instrument × method)* run, the harness reports:

**Hard-constraint metrics (playability):**
- `hard_feasible`: output satisfies **range** and **polyphony** constraints for the target instrument.
- `status`: `ACCEPTED`, `CORRECTED`, or `REJECTED` (system behavior).

**Soft-constraint metrics (comfort):**
- `tessitura_violations`: number of notes outside the preferred register.

**Change magnitude:**
- `shift`: semitone shift applied (global octave shifts in this prototype).
- `abs_shift`: `|shift|` (how far the music moved).

We also summarize feasibility and tessitura per instrument and method.

## Representative instruments per category (used in this notebook)

- **vocal:** Tenor Voice (`TENOR_VOICE`)
- **woodwind:** Flute (`FLUTE`), Clarinet (`CLARINET`)
- **brasswind:** Trumpet (`TRUMPET`)
- **strings:** Violin (`VIOLIN`)
- **percussion (pitched):** Marimba (`MARIMBA`)

In [ ]:
# hard feasibility metric: check returned notes against hard constraints

def hard_feasible(notes_out, profile):
    """True playability: range + polyphony must be satisfied."""
    if notes_out is None:
        return False
    if out_of_range(notes_out, profile):
        return False
    if max_simultaneous_notes(notes_out) > profile["max_polyphony"]:
        return False
    return True

# Re run harness results with a clean hard_feasible flag
df_results2 = df_results.copy()

# Reconstruct output notes from the stored shift (global shifts only)
def reconstruct_output_notes(row):
    base_notes = EVAL_CLIPS[row["clip"]]
    shift = row["shift"]
    if shift is None:
        return None
    try:
        shift = int(shift)
    except Exception:
        return None
    return shift_notes(base_notes, shift)

df_results2["output_notes"] = df_results2.apply(reconstruct_output_notes, axis=1)
df_results2["hard_feasible"] = df_results2.apply(lambda r: hard_feasible(r["output_notes"], EVAL_INSTRUMENTS[r["instrument"]]), axis=1)
df_results2["abs_shift"] = df_results2["shift"].abs()

wide_hard = df_results2.pivot_table(
    index=["clip","instrument","category"],
    columns="method",
    values=["hard_feasible","tessitura_violations","abs_shift","status"],
    aggfunc="first"
)
wide_hard.columns = [f"{a}_{b}" for a,b in wide_hard.columns]
wide_hard = wide_hard.reset_index()

wide_hard.head(20)

In [ ]:
import numpy as np

summary = (
    df_results2.groupby(["instrument","category","method"])
    .agg(
        hard_feasible_rate=("hard_feasible","mean"),
        avg_tessitura=("tessitura_violations","mean"),
        avg_abs_shift=("abs_shift","mean"),
        accepted=("status", lambda s: np.mean(s=="ACCEPTED")),
        corrected=("status", lambda s: np.mean(s=="CORRECTED")),
        rejected=("status", lambda s: np.mean(s=="REJECTED")),
        n=("status","count"),
    )
    .reset_index()
    .sort_values(["category","instrument","method"])
)

summary

In [ ]:
# Statistical tests to check if advanced is working

import numpy as np
import scipy.stats as stats

def mcnemar_exact(a_bool, b_bool):
    a = np.asarray(a_bool, dtype=bool)  # advanced
    b = np.asarray(b_bool, dtype=bool)  # simple
    n01 = np.sum((~a) & (b))  # advanced false, simple true
    n10 = np.sum((a) & (~b))  # advanced true, simple false
    n00 = np.sum((~a) & (~b))
    n11 = np.sum((a) & (b))
    n = n01 + n10
    p = 2 * stats.binomtest(min(n01, n10), n=n, p=0.5).pvalue if n > 0 else 1.0
    return n00, n01, n10, n11, p

# 1) Hard feasibility: McNemar test
n00, n01, n10, n11, p_mcnemar = mcnemar_exact(
    wide_hard["hard_feasible_constraint_aware"],
    wide_hard["hard_feasible_simple_pitch"],
)

print("McNemar contingency [n00 n01 n10 n11]:", [n00, n01, n10, n11])
print("McNemar exact p-value:", p_mcnemar)

# 2) Soft metric: tessitura violations (paired), only where both are hard feasible
both_feasible = wide_hard[(wide_hard["hard_feasible_constraint_aware"]==True) & (wide_hard["hard_feasible_simple_pitch"]==True)].copy()
print("Num paired cases where both methods are hard-feasible:", len(both_feasible))

if len(both_feasible) > 1:
    delta_tess = both_feasible["tessitura_violations_constraint_aware"] - both_feasible["tessitura_violations_simple_pitch"]
    t_res = stats.ttest_rel(both_feasible["tessitura_violations_constraint_aware"], both_feasible["tessitura_violations_simple_pitch"])
    w_res = None if np.all(delta_tess==0) else stats.wilcoxon(delta_tess, alternative="two-sided")

    print("Mean tessitura (simple):", both_feasible["tessitura_violations_simple_pitch"].mean())
    print("Mean tessitura (advanced):", both_feasible["tessitura_violations_constraint_aware"].mean())
    print("Mean delta (advanced - simple):", delta_tess.mean())
    print("Paired t-test p:", t_res.pvalue)
    print("Wilcoxon p:", (w_res.pvalue if w_res is not None else None))

    # 3) Change magnitude: abs shift (paired)
    delta_abs_shift = both_feasible["abs_shift_constraint_aware"] - both_feasible["abs_shift_simple_pitch"]
    t_shift = stats.ttest_rel(both_feasible["abs_shift_constraint_aware"], both_feasible["abs_shift_simple_pitch"])
    w_shift = None if np.all(delta_abs_shift==0) else stats.wilcoxon(delta_abs_shift, alternative="two-sided")

    print("Mean |shift| (simple):", both_feasible["abs_shift_simple_pitch"].mean())
    print("Mean |shift| (advanced):", both_feasible["abs_shift_constraint_aware"].mean())
    print("Mean delta |shift| (advanced - simple):", delta_abs_shift.mean())
    print("Paired t-test p (|shift|):", t_shift.pvalue)
    print("Wilcoxon p (|shift|):", (w_shift.pvalue if w_shift is not None else None))

**Hard feasibility (playability):**  
Both methods produced identical feasibility outcomes (McNemar p = 1.0).  
This shows the advanced method does not create playable results where none existed; instead, it selects among already playable mappings.

**Comfort (tessitura):**  
The constraint aware method significantly reduced tessitura violations (paired t-test p = 0.0167, Wilcoxon p = 0.0256).  
This indicates improved ergonomic playability through better register selection.

**Magnitude of change:**  
The advanced method did not significantly increase pitch movement (|shift| p = 0.10), suggesting improvements are due to smarter octave choice rather than excessive modification.

**Conclusion:**  
The constraint aware correction functions as a refinement mechanism rather than a full repair. It preserves feasibility while improving the comfort of the resulting music.

In [ ]:
# Add GM MIDI program numbers to profiles (0-127) for export/listening
# General MIDI programs (1-128). MIDI messages use 0-127.
# Flute=74->73, Clarinet=72->71, Trumpet=57->56, Violin=41->40, Marimba=13->12, Choir Aahs=53->52

INSTRUMENT_PROFILES["FLUTE"]["midi_program"] = 73
INSTRUMENT_PROFILES["CLARINET"]["midi_program"] = 71
INSTRUMENT_PROFILES["TRUMPET"]["midi_program"] = 56
INSTRUMENT_PROFILES["VIOLIN"]["midi_program"] = 40
INSTRUMENT_PROFILES["MARIMBA"]["midi_program"] = 12
INSTRUMENT_PROFILES["TENOR_VOICE"]["midi_program"] = 52

In [ ]:
pip install mido

In [ ]:
import os
from pathlib import Path
import mido

# We represent time in beats, not seconds.
# This avoids tempo handling and keeps the prototype simple.

def midi_to_notes(path):
    """
    Read a MIDI file and extract notes as (pitch, start_beats, duration_beats).
    """
    mid = mido.MidiFile(path)
    tpb = mid.ticks_per_beat

    merged = mido.merge_tracks(mid.tracks)

    active = {} 
    notes = []
    time_ticks = 0

    for msg in merged:
        time_ticks += msg.time
        if msg.type not in ("note_on", "note_off"):
            continue

        pitch = msg.note
        t_beats = time_ticks / tpb

        is_note_off = (msg.type == "note_off") or (msg.type == "note_on" and getattr(msg, "velocity", 0) == 0)

        if not is_note_off:
            active.setdefault(pitch, []).append(t_beats)
        else:
            if pitch in active and len(active[pitch]) > 0:
                start = active[pitch].pop(0)
                dur = max(0.0, t_beats - start)
                notes.append((int(pitch), float(start), float(dur)))

    notes.sort(key=lambda x: (x[1], x[0]))
    return notes


def notes_to_midi(notes, out_path, program=0, bpm=120, ticks_per_beat=480):
    """
    Write notes (pitch, start_beats, duration_beats) to a MIDI file.
    - Single track
    - Inserts tempo + program_change at time 0
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    mid = mido.MidiFile(ticks_per_beat=ticks_per_beat)
    track = mido.MidiTrack()
    mid.tracks.append(track)

    tempo = mido.bpm2tempo(bpm)
    track.append(mido.MetaMessage("set_tempo", tempo=tempo, time=0))
    track.append(mido.Message("program_change", program=int(program), time=0))

    events = []
    for pitch, start, dur in notes:
        start_tick = int(round(start * ticks_per_beat))
        end_tick = int(round((start + dur) * ticks_per_beat))
        end_tick = max(end_tick, start_tick + 1)
        events.append((start_tick, "on", int(pitch)))
        events.append((end_tick, "off", int(pitch)))

    events.sort(key=lambda e: (e[0], 0 if e[1] == "off" else 1, e[2]))

    last_tick = 0
    for tick, typ, pitch in events:
        delta = tick - last_tick
        last_tick = tick
        if typ == "on":
            track.append(mido.Message("note_on", note=pitch, velocity=64, time=delta))
        else:
            track.append(mido.Message("note_off", note=pitch, velocity=64, time=delta))

    track.append(mido.MetaMessage("end_of_track", time=0))
    mid.save(str(out_path))
    return str(out_path)


def patch_only_baseline(in_path, out_path, program=0):
    """
    Baseline: keep the MIDI the same, but change the instrument patch.
    """
    in_path = Path(in_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    mid = mido.MidiFile(str(in_path))
    for track in mid.tracks:
        new_msgs = []
        inserted = False
        for msg in track:
            if msg.type == "program_change":
                continue
            if not inserted and not msg.is_meta:
                new_msgs.append(mido.Message("program_change", program=int(program), time=0))
                inserted = True
            new_msgs.append(msg)
        if not inserted:
            new_msgs.insert(0, mido.Message("program_change", program=int(program), time=0))
        track[:] = new_msgs

    mid.save(str(out_path))
    return str(out_path)

In [ ]:
import pandas as pd
from pathlib import Path

FILE_METHODS = {
    "patch_only": None,              
    "simple_pitch": simple_pitch_correction,
    "constraint_aware": constraint_aware_correction,
}

def run_file_harness(dataset_dir, instruments, methods=FILE_METHODS, outputs_dir="outputs", bpm=120):
    """
    Process a folder of MIDI files.
    For each file x instrument x method:
    - read midi to notes
    - write MIDI with patch change only
    - run correction to write MIDI output
    Returns a results DataFrame with metrics.
    """
    dataset_dir = Path(dataset_dir)
    outputs_dir = Path(outputs_dir)
    outputs_dir.mkdir(parents=True, exist_ok=True)

    midi_files = sorted([p for p in dataset_dir.glob("**/*") if p.suffix.lower() in (".mid", ".midi")])
    if not midi_files:
        raise FileNotFoundError(f"No .mid/.midi files found under: {dataset_dir}")

    rows = []
    for midi_path in midi_files:
        clip_name = midi_path.stem
        base_notes = midi_to_notes(midi_path)

        for inst_key, profile in instruments.items():
            program = int(profile.get("midi_program", 0))

            for method_name, fn in methods.items():
                out_folder = outputs_dir / inst_key / method_name
                out_folder.mkdir(parents=True, exist_ok=True)
                out_path = out_folder / f"{clip_name}.mid"

                if method_name == "patch_only":
                    patch_only_baseline(midi_path, out_path, program=program)
                    out_notes = base_notes
                    status = "UNCHANGED"
                    reason = "Patch-only baseline (no symbolic changes)"
                    shift = 0
                    tv = tessitura_violations(out_notes, profile)
                    hf = hard_feasible(out_notes, profile)

                else:
                    res = fn(base_notes, profile)
                    out_notes = res["notes"]
                    status = res["status"]
                    reason = res["reason"]
                    shift = res.get("shift", 0)

                    if out_notes is None:
                        patch_only_baseline(midi_path, out_path, program=program)
                        tv = tessitura_violations(base_notes, profile)
                        hf = False
                    else:
                        notes_to_midi(out_notes, out_path, program=program, bpm=bpm)
                        tv = tessitura_violations(out_notes, profile)
                        hf = hard_feasible(out_notes, profile)

                rows.append({
                    "file": str(midi_path),
                    "clip": clip_name,
                    "instrument": inst_key,
                    "category": profile.get("category", ""),
                    "method": method_name,
                    "status": status,
                    "hard_feasible": hf,
                    "tessitura_violations": tv,
                    "shift": shift,
                    "abs_shift": abs(shift) if shift is not None else None,
                    "output_midi": str(out_path),
                    "reason": reason
                })

    return pd.DataFrame(rows)

# Demo: generate one MIDI from an existing symbolic clip, then run the file harness
demo_dir = Path("dataset_demo")
demo_dir.mkdir(exist_ok=True)

demo_in = demo_dir / "demo_input.mid"
demo_notes = EVAL_CLIPS["flute_out_of_range"]
notes_to_midi(demo_notes, demo_in, program=0, bpm=120)

print("Wrote demo input MIDI:", demo_in)

df_file = run_file_harness(demo_dir, EVAL_INSTRUMENTS, outputs_dir="outputs_demo", bpm=120)
df_file.sort_values(["clip","instrument","method"]).head(30)

In [ ]:
file_feas = (
    df_file.groupby(["instrument","method"])["hard_feasible"]
    .mean()
    .reset_index()
    .rename(columns={"hard_feasible":"hard_feasible_rate"})
)

file_tess = (
    df_file.groupby(["instrument","method"])["tessitura_violations"]
    .mean()
    .reset_index()
    .rename(columns={"tessitura_violations":"avg_tessitura_violations"})
)

file_feas.merge(file_tess, on=["instrument","method"], how="left").sort_values(["instrument","method"])

## Summary
Quick aggregate views for feasibility rate and tessitura comfort.


In [ ]:
feas_rate = (
    df_results.groupby(["instrument", "method"])["playable"]
    .mean()
    .reset_index()
    .rename(columns={"playable": "feasible_rate"})
)

avg_tess = (
    df_results[df_results["playable"]]
    .groupby(["instrument", "method"])["tessitura_violations"]
    .mean()
    .reset_index()
    .rename(columns={"tessitura_violations": "avg_tessitura_violations"})
)

feas_rate.merge(avg_tess, on=["instrument", "method"], how="left").sort_values(["instrument", "method"])